In [1]:
class BTreeNode:
    def __init__(self, leaf=True):
        self.leaf = leaf
        self.keys = []
        self.children = []

class BTree:
    def __init__(self, m=3):
        self.root = BTreeNode(leaf=True)
        self.m = m                     # 阶数，最大孩子数 = m
        self.max_keys = m - 1          # 最多 m-1 个键，这里是 2
        # 非根节点最少键数（3阶时为1）
        self.min_keys = (m + 1) // 2 - 1 if m % 2 == 0 else (m // 2)  # 3阶 => 1

    def insert(self, key):
        # 辅助函数：向节点 node 插入键，返回 (是否发生分裂, 上移的键, 新右节点)
        def _insert(node, k):
            if node.leaf:
                # 叶子节点：直接插入键（可能变成3个）
                node.keys.append(k)
                node.keys.sort()
                if len(node.keys) <= self.max_keys:
                    return (False, None, None)
                # 需要分裂：节点有3个键，取中间键上移
                mid = len(node.keys) // 2          # 索引1（0,1,2）
                left_keys = node.keys[:mid]
                right_keys = node.keys[mid+1:]
                up_key = node.keys[mid]
                # 创建左右孩子（都是叶子）
                left_node = BTreeNode(leaf=True)
                left_node.keys = left_keys
                right_node = BTreeNode(leaf=True)
                right_node.keys = right_keys
                return (True, up_key, (left_node, right_node))
            else:
                # 内部节点：找到合适的子节点递归插入
                i = 0
                while i < len(node.keys) and k > node.keys[i]:
                    i += 1
                # 递归向 child = node.children[i] 插入
                has_split, up_key, new_nodes = _insert(node.children[i], k)
                if not has_split:
                    return (False, None, None)
                # 子节点分裂了，需要将 up_key 插入当前节点，并用 new_nodes 替换原子节点
                left_child, right_child = new_nodes
                # 删除原来的 child，插入 left_child 和 right_child
                node.children.pop(i)
                node.children.insert(i, left_child)
                node.children.insert(i+1, right_child)
                node.keys.insert(i, up_key)
                if len(node.keys) <= self.max_keys:
                    return (False, None, None)
                # 当前节点也满了（达到3个键），继续分裂
                mid = len(node.keys) // 2
                left_keys = node.keys[:mid]
                right_keys = node.keys[mid+1:]
                up_key2 = node.keys[mid]
                left_node = BTreeNode(leaf=False)
                left_node.keys = left_keys
                left_node.children = node.children[:mid+1]
                right_node = BTreeNode(leaf=False)
                right_node.keys = right_keys
                right_node.children = node.children[mid+1:]
                return (True, up_key2, (left_node, right_node))

        # 从根开始插入
        has_split, up_key, new_nodes = _insert(self.root, key)
        if not has_split:
            return
        # 根节点分裂，创建新根
        left_child, right_child = new_nodes
        new_root = BTreeNode(leaf=False)
        new_root.keys = [up_key]
        new_root.children = [left_child, right_child]
        self.root = new_root

    def inorder(self):
        res = []
        def _inorder(node):
            for i in range(len(node.keys)):
                if not node.leaf:
                    _inorder(node.children[i])
                res.append(node.keys[i])
            if not node.leaf:
                _inorder(node.children[-1])
        _inorder(self.root)
        return res

    # 打印树（使用你原来的打印方法，略微调整以支持没有键的情况）
    def _node_label(self, node):
        return f"[{','.join(map(str, node.keys))}]"

    def _build_tree_lines(self, node):
        if not node or (not node.keys and not node.children):
            return [], 0, 0, 0
        label = self._node_label(node)
        label_len = len(label)
        child_count = len(node.children)
        if child_count == 0:
            return [label], label_len, 1, label_len // 2
        child_data = [self._build_tree_lines(ch) for ch in node.children]
        child_lines = [cd[0] for cd in child_data]
        child_widths = [cd[1] for cd in child_data]
        child_heights = [cd[2] for cd in child_data]
        child_centers = [cd[3] for cd in child_data]
        gap = 2
        total_child_width = sum(child_widths) + gap * (child_count - 1)
        parent_center = total_child_width // 2
        label_start = max(0, parent_center - label_len // 2)
        first_line = ' ' * label_start + label + ' ' * (total_child_width - label_start - label_len)
        second_line_chars = [' '] * total_child_width
        cur_x = 0
        for i, (ch_width, ch_center) in enumerate(zip(child_widths, child_centers)):
            child_center_global = cur_x + ch_center
            if child_count == 1:
                symbol = '/'
            else:
                if i == 0:
                    symbol = '/'
                elif i == child_count - 1:
                    symbol = '\\'
                else:
                    symbol = '\\'
            if 0 <= child_center_global < total_child_width:
                second_line_chars[child_center_global] = symbol
            cur_x += ch_width + gap
        second_line = ''.join(second_line_chars).rstrip()
        if not second_line:
            second_line = ' ' * total_child_width
        else:
            second_line = second_line.ljust(total_child_width)
        max_child_height = max(child_heights) if child_heights else 0
        merged_lines = []
        for row in range(max_child_height):
            line_parts = []
            cur_x = 0
            for i, (ch_lines, ch_width) in enumerate(zip(child_lines, child_widths)):
                if row < len(ch_lines):
                    line_parts.append(ch_lines[row])
                else:
                    line_parts.append(' ' * ch_width)
                if i < child_count - 1:
                    line_parts.append(' ' * gap)
            merged_lines.append(''.join(line_parts))
        all_lines = [first_line, second_line] + merged_lines
        final_width = max(len(line) for line in all_lines)
        all_lines = [line.ljust(final_width) for line in all_lines]
        return all_lines, final_width, len(all_lines), label_start + label_len // 2

    def print_tree(self):
        if not self.root.keys and not self.root.children:
            print("空树")
            return
        lines, _, _, _ = self._build_tree_lines(self.root)
        for line in lines:
            print(line)

    # 验证 B-Tree 性质
    def verify(self):
        props = []
        leaf_depth = None
        def check_depth(node, d):
            nonlocal leaf_depth
            if node.leaf:
                if leaf_depth is None:
                    leaf_depth = d
                elif leaf_depth != d:
                    return False
                return True
            for ch in node.children:
                if not check_depth(ch, d+1):
                    return False
            return True
        same_depth = check_depth(self.root, 0)
        props.append(("所有叶子节点在同一层", same_depth))
        root_ok = 1 <= len(self.root.keys) <= self.max_keys
        props.append(("根节点键数在 [1, max_keys] 内", root_ok))
        def check_non_root(node, is_root):
            if is_root:
                return True
            return self.min_keys <= len(node.keys) <= self.max_keys
        def dfs(node, is_root):
            if not check_non_root(node, is_root):
                return False
            for ch in node.children:
                if not dfs(ch, False):
                    return False
            return True
        nonroot_ok = dfs(self.root, True)
        props.append(("所有非根节点键数在 [min_keys, max_keys] 内", nonroot_ok))
        inorder = self.inorder()
        sorted_ok = all(inorder[i] <= inorder[i+1] for i in range(len(inorder)-1))
        props.append(("中序遍历严格升序", sorted_ok))
        def check_child_count(node):
            if not node.leaf:
                if len(node.children) != len(node.keys) + 1:
                    return False
                for ch in node.children:
                    if not check_child_count(ch):
                        return False
            return True
        child_ok = check_child_count(self.root)
        props.append(("每个内部节点的子节点数 = 键数+1", child_ok))
        return props


if __name__ == "__main__":
    seq = [10, 20, 5, 6, 12, 30, 25]
    btree = BTree(m=3)

    print("3阶 B-Tree 构建过程 (2-3树)")
    print(f"插入序列: {seq}\n")
    for val in seq:
        print(f"插入 {val}:")
        btree.insert(val)
        btree.print_tree()
        print("-" * 40)

    print("\n========== 最终 B-Tree ==========")
    btree.print_tree()

    print("\n========== 性质验证 ==========")
    for desc, ok in btree.verify():
        print(f"✓ {desc}: {'通过' if ok else '不通过'}")

    print(f"\n中序遍历序列: {btree.inorder()}")

3阶 B-Tree 构建过程 (2-3树)
插入序列: [10, 20, 5, 6, 12, 30, 25]

插入 10:
[10]
----------------------------------------
插入 20:
[10,20]
----------------------------------------
插入 5:
  [10]   
 /     \ 
[5]  [20]
----------------------------------------
插入 6:
   [10]    
  /      \ 
[5,6]  [20]
----------------------------------------
插入 12:
     [10]     
  /       \   
[5,6]  [12,20]
----------------------------------------
插入 30:
     [10,20]     
  /      \     \ 
[5,6]  [12]  [30]
----------------------------------------
插入 25:
       [10,20]      
  /      \      \   
[5,6]  [12]  [25,30]
----------------------------------------

========== 最终 B-Tree ==========
       [10,20]      
  /      \      \   
[5,6]  [12]  [25,30]

========== 性质验证 ==========
✓ 所有叶子节点在同一层: 通过
✓ 根节点键数在 [1, max_keys] 内: 通过
✓ 所有非根节点键数在 [min_keys, max_keys] 内: 通过
✓ 中序遍历严格升序: 通过
✓ 每个内部节点的子节点数 = 键数+1: 通过

中序遍历序列: [5, 6, 10, 12, 20, 25, 30]
